I found it hard to use Docent as a system of record for my judge iteration experiments. I frequently tended to store data locally, due to issues with uploading and downloading data to the Docent backend.Here’s a first pass at a new DX which aims to address that.

To start: imagine you already have a bunch of agent_runs sitting in a collection. 

In [ ]:
from docent import Docent

client = Docent()
active_collection_id = "4707073b-2e3b-444b-a2e1-5248beaaaa62"

One key part of the experience we want to improve is storing labels in Docent. First, a user should create a labelset with a specified schema.

In [ ]:
from typing import Any
from docent import LabelSet, Label

labelset = LabelSet(
    name="swe_bench_sandbox",
    description="Label set for the SWE Bench Sandbox",
    schema={...},
)
client.create_labelset(labelset)  # Pushes the labelset to the backend

# At this point, the labelset should show up in the UI

Next, the user has to create a correspondence between each label and the agent_run that it labels; then, they can register the label. This part will be manual and bespoke because everyone will have their raw_labels in a different format. 

In [ ]:
raw_labels: list[dict[str, Any]] = [
    {
        "key": {
            "model_name": "20250805-openhands-Qwen3-Coder-30B-A3B-Instruct",
            "instance_id": "django__django-12965",
        },
        "value": {
            "label": "cheating",
            "explanation": "The agent cheated",
        }
    },
]

for rl in raw_labels:
    agent_run_ids = client.execute_dql("select id from agent_runs where metadata ...")
    if len(agent_run_ids) != 1:
        raise ValueError(f"Expected 1 agent run id, got {len(agent_run_ids)}")
    client.add_label(labelset.id, Label(rl["value"]))

# At this point, the UI should show all the labels

Now, the user could create a judge in the UI and iterate that way. But we also want to allow people to run and iterate on the judge locally, which has benefits such as allowing you to specify precisely what AgentRuns you want the judge to run over, how many times it's run, etc (as opposed to the hardcoded defaults in the UI).

In [ ]:
from docent import Judge

# If you're creating a new judge:
judge = Judge()
client.add_judge(judge)
# If you're re-using an existing judge:
judge = docent.ls_judges()[0]
judge = docent.get_judge_by_name("asdf")

In [ ]:
# Now run the judge on the collection, but only runs with labels
agent_run_ids = client.execute_dql(
    f"select id from agent_runs inner join labels where labels.labelset_id = '{labelset.id}'"
)
agent_runs = client.get_agent_runs(agent_run_ids)

for agent_run in agent_runs:
    for _ in range(10):
        judge(agent_run)  # Auto-logs JudgeResults and metadata back into docent
        # TODO(mengk): do judges deserve their own collection?
        # TODO(mengk): should we allow people to run their judge on any text, not just AgentRuns?
        # TODO(mengk): what if they pass an agent run that doesn't exist within Docent?

#  At this point, the judge results page shows all the results like it does today

Next, we want to flexibly do quantitative analysis and connect that to qualitative insights.

For the quantitative part: you should be able to use DQL to extract whatever you want from the database and throw it into a DataFrame. This puts data scientists back in an environment where they're comfortable.

Code is helpful for tackling the long tail of analysis questions that are hard to fully support in the UI, particularly when they involve connections between AgentRun metadata, judge results, and labels. For example: if I have two versions of an AI agent where the second one was derived from the first, how do I isolate examples where version 2 did better vs worse than version 1? Can I use a judge to understand why?

In [ ]:
import pandas as pd

# now i want to do analysis
# agent_run_id, judge_id, judge_result (dict), judge_metadata (dict), labelset_id, label (dict),
analysis_data: list[dict[str, Any]] = client.dql("give me the cols above")

df = pd.DataFrame(analysis_data)

# i write a function to do group by
df2 = df.groupby("agent_run_id").agg({
    "judge_result": "first",
    "judge_metadata": "first",
    "labelset_id": "first",
    "label": "first",
    "judge_result_id": list,
    # computed data from agg
    "entropy": None,
    "p_correct": None,
    "log_loss": None,
    "argmax_judge_result": None,
    "argmax_correct": None,
    "gold_label": None,
    "distr": None,
    "distr_entropy": None,
    "distr_p_correct": None,
    "distr_log_loss": None,
})

# Look at the worst-performing results, when averaged over 5 rollouts
df2 = df2.sort_values(by="p_correct", ascending=True)

Next up is connecting the quantitative analysis to the qualitative. I might want to look qualitatively at the worst performing judge runs and understand how the different rollouts differed.

In [ ]:
for index, row in df2.iterrows():  # Go row by row, worst performing results first
    cur_judge_result_ids = row["judge_result_id"]  # Get the list of judge outputs
    client.give_me_links(cur_judge_result_ids)  # Open the judge results URL for easy viewing in UI

    # More speculative: making it easy to chat with the rollouts
    # TODO(mengk): how could we support citations and easy viewing in the UI?
    all_rollout_texts = []
    for id in cur_judge_result_ids:
        # this is json object contains everything about the judge result
        judge_reuslt_obj = client.get_judge_result(id)
        all_rollout_texts.append(str(judge_reuslt_obj))

    client.ask_question(f"""what are idfferences between these rollouts: {all_rollout_texts}""")

Some other things that weren't discussed here:
* Computed or derived fields where someone wants to update the AgentRun metadata with something new